<a href="https://colab.research.google.com/github/PMQ9/Mixture-of-Experts_Research/blob/main/colab_training_verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mixture-of-Experts: Training & Verification on Google Colab

This notebook sets up training and formal verification of MoE experts on Google Colab with free GPU.

**Features:**
- GPU acceleration (T4 or A100)
- Google Drive integration for persistent storage
- Training individual experts (MNIST, CIFAR-10, GTSRB)
- Formal verification with alpha-beta-CROWN
- Support for both NRT (Non-Robust Training) and AT (Adversarial Training)

**Setup time:** 5-10 minutes

## Step 1: Setup GPU & Check Environment

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

work_dir = '/content/drive/MyDrive/MoE_Research'
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

print(f"Working directory: {os.getcwd()}")

## Step 3: Clone Repository

In [ ]:
import subprocess
from pathlib import Path

repo_url = "https://github.com/PMQ9/Mixture-of-Experts_Research.git"
repo_dir = Path(work_dir) / "Mixture-of-Experts_Research"

if not repo_dir.exists():
    print(f"Cloning repository...")
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
    print("Repository cloned successfully!")
else:
    print(f"Repository already exists at {repo_dir}")
    os.chdir(repo_dir)
    print("Updating repository...")
    subprocess.run(["git", "pull"], check=True)

os.chdir(repo_dir)
print(f"Working directory: {os.getcwd()}")
!ls -la | head -20

## Step 4: Install Dependencies

In [ ]:
print("Installing PyTorch and dependencies...")
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install tqdm matplotlib netron onnx timm scipy opencv-python adversarial-robustness-toolbox -q
print("Dependencies installed!")

In [ ]:
print("Installing alpha-beta-CROWN...")
import os
os.chdir("/content/drive/MyDrive/MoE_Research/Mixture-of-Experts_Research")

!pip install git+https://github.com/Verified-Intelligence/auto_LiRPA.git -q
!pip uninstall onnx2pytorch -y -q
!pip install git+https://github.com/Verified-Intelligence/onnx2pytorch.git -q

print("alpha-beta-CROWN dependencies installed!")

## Step 5: Verify Setup

In [ ]:
import torch
from pathlib import Path

print("=" * 80)
print("COLAB ENVIRONMENT SETUP VERIFICATION")
print("=" * 80)

print(f"\n1. GPU Status")
print(f"   Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print(f"\n2. Repository Files")
for item in sorted(Path.cwd().iterdir())[:10]:
    print(f"   - {item.name}")

print(f"\n3. Key Directories")
for dir_path in ["src/Vision_Transformer_Pytorch", "src/Formal_Neural_Network_Verification", "artifacts"]:
    exists = Path(dir_path).exists()
    status = "OK" if exists else "MISSING"
    print(f"   {dir_path}: {status}")

print(f"\nSetup complete!")

# SECTION: Training Experts

## Train Individual Experts

Select a dataset and run a training cell. Typical training: 30-60 minutes on Colab GPU.

In [ ]:
import os
from pathlib import Path

os.chdir(Path.cwd())

dataset = "MNIST"
model_arch = "ultra_verifiable_cnn"
epochs = 100
batch_size = 128

print(f"Training {dataset} expert (Non-Robust)")
print(f"Epochs: {epochs}, Batch size: {batch_size}\n")

!python train.py --dataset {dataset} --model_arch {model_arch} --epochs {epochs} --batch_size {batch_size} --device cuda

In [ ]:
import os
os.chdir(Path.cwd())

dataset = "MNIST"
model_arch = "ultra_verifiable_cnn"
epochs = 100

print(f"Training {dataset} expert (Adversarial Training - PGD)\n")

!python train.py --dataset {dataset} --model_arch {model_arch} --epochs {epochs} --batch_size 128 --device cuda --adv_training --at_mode PGD

In [ ]:
import os
os.chdir(Path.cwd())

dataset = "CIFAR10"
model_arch = "ultra_verifiable_cnn"
epochs = 100

print(f"Training {dataset} expert (Non-Robust)\n")

!python train.py --dataset {dataset} --model_arch {model_arch} --epochs {epochs} --batch_size 128 --device cuda

In [ ]:
import os
os.chdir(Path.cwd())

dataset = "CIFAR10"
model_arch = "ultra_verifiable_cnn"
epochs = 100

print(f"Training {dataset} expert (Adversarial Training - PGD)\n")

!python train.py --dataset {dataset} --model_arch {model_arch} --epochs {epochs} --batch_size 128 --device cuda --adv_training --at_mode PGD

# SECTION: Formal Verification with alpha-beta-CROWN

## Verify Expert Models

Use alpha-beta-CROWN (state-of-the-art verifier) to formally verify robustness.

In [ ]:
import os
os.chdir(Path.cwd())

model_path = "artifacts/mnist_ultra_verifiable_cnn_best_og.pth"
dataset = "MNIST"
epsilon = 0.00784
num_images = 10
timeout = 300

print(f"Verifying: {model_path}")
print(f"Dataset: {dataset}, Epsilon: {epsilon:.5f} ({epsilon*255:.1f}/255)")
print(f"Images: {num_images}, Est. time: {num_images * timeout / 60:.1f} min\n")

!python src/Formal_Neural_Network_Verification/verify_expert_abcrown.py --model_path {model_path} --dataset {dataset} --epsilon {epsilon} --num_images {num_images} --timeout {timeout}

In [ ]:
import os
os.chdir(Path.cwd())

model_path = "artifacts/meta_moe_ultra_verifiable_cnn_best_og.pth"
num_mnist = 5
num_cifar = 5

print(f"Verifying MetaMoE Router: {model_path}")
print(f"MNIST samples: {num_mnist}, CIFAR-10 samples: {num_cifar}")
print(f"Est. time: {(num_mnist + num_cifar) * 300 / 60:.1f} min\n")

!python verify_all_router_samples.py --model_path {model_path} --num_mnist {num_mnist} --num_cifar {num_cifar} --timeout 300

# SECTION: Results & Download

## View Results

In [ ]:
from pathlib import Path
import os

os.chdir(Path.cwd())

print("Trained models:")
artifacts_dir = Path("artifacts")
if artifacts_dir.exists():
    for pth_file in sorted(artifacts_dir.glob("*.pth")):
        size_mb = pth_file.stat().st_size / 1e6
        print(f"  {pth_file.name} ({size_mb:.1f} MB)")
else:
    print("  No artifacts directory found")

In [ ]:
from google.colab import files
from pathlib import Path
import os

os.chdir(Path.cwd())

files_to_download = [
    "expert_verification_results.txt",
    "artifacts/mnist_ultra_verifiable_cnn_best_og.pth",
    "artifacts/cifar10_ultra_verifiable_cnn_best_og.pth",
]

print("Downloading files...\n")

for file_path in files_to_download:
    full_path = Path(file_path)
    if full_path.exists():
        print(f"Downloading: {file_path}")
        files.download(file_path)
    else:
        print(f"Skipping (not found): {file_path}")

print("\nDownload complete!")

## Tips & Troubleshooting

### Setup
- **GPU**: Go to Runtime > Change runtime type > Select GPU (T4 or A100)
- **Storage**: Models saved to Google Drive (15GB free, persistent)
- **Setup time**: 10-15 minutes (mostly installing dependencies)

### Training Time
- **Expert training** (100 epochs): 30-45 minutes on T4
- **With Adversarial Training**: 60-90 minutes
- **Batch size**: Reduce to 64 if OOM errors occur

### Verification Time
- **Per image** at epsilon=2/255: 10-15 seconds
- **10 images**: 2-3 minutes
- **100 images**: 20-30 minutes (recommended for papers)
- **Session timeout**: 12 hours (save frequently to Drive)

### Troubleshooting

**"GPU not available"**
- Runtime > Change runtime type > GPU

**"CUDA out of memory"**
- Reduce batch_size to 64
- Use smaller model_arch (tiny_cnn)

**"ModuleNotFoundError"**
- Re-run installation cells (Step 4)
- Runtime > Restart session

**"Verification timeout"**
- Increase timeout parameter to 600
- Reduce num_images

### Tips
- Save models frequently to Google Drive
- Download results after verification completes
- Run 1-2 cells at a time to monitor progress
- Clear cell output if memory builds up